# minigrpo and selfimprove on a T4

Use a GPU runtime. The first script trains GRPO on the integer seed-split train rows and scores dev. The second proposes recomputable problems, trains with that GRPO, and rolls a candidate back through `PromotionGate`. Confirm is not scored.

Outputs and the Hugging Face cache live on Drive. Re-running a cell continues the same directory. `--check` does not train.

The default checkpoint is Qwen2.5-0.5B-Instruct, revision `7ae557604adf67be50417f59c2c2f167def9a775`. A T4 can also load Qwen2.5-3B-Instruct revision `aa8e72537993ba99e69dfaafa59ed015b17504d1` by changing `MODEL` and `REVISION`. This PyTorch path is not a numerical replicate of the MLX path. No model result is stored in the repository.


In [ ]:
import os

os.environ.setdefault("VARE_REPO", "https://github.com/mitukx/VARE.git")
os.environ.setdefault("VARE_REV", "cursor/minigrpo-colab-af75")
os.environ.setdefault("MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
os.environ.setdefault("REVISION", "7ae557604adf67be50417f59c2c2f167def9a775")
print(os.environ["VARE_REPO"], os.environ["VARE_REV"])
print(os.environ["MODEL"], os.environ["REVISION"])


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/vare-minigrpo")
if not (repo / ".git").exists():
    subprocess.check_call(["git", "clone", os.environ["VARE_REPO"], str(repo)])
subprocess.check_call(["git", "fetch", "origin", os.environ["VARE_REV"]], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "/content/vare-minigrpo",
        "transformers",
        "accelerate",
        "peft",
        "huggingface_hub",
    ]
)


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/vare")
root.mkdir(parents=True, exist_ok=True)
minigrpo_output = root / "minigrpo-torch"
selfimprove_output = root / "selfimprove-torch"
minigrpo_output.mkdir(parents=True, exist_ok=True)
selfimprove_output.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(root / "huggingface-cache")
os.environ["HF_HUB_CACHE"] = str(root / "huggingface-cache" / "hub")
print("minigrpo", minigrpo_output)
print("selfimprove", selfimprove_output)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-minigrpo")
for script in ("scripts/run_minigrpo.py", "scripts/run_selfimprove.py"):
    checked = subprocess.run(
        [sys.executable, str(repo / script), "--backend", "torch", "--check"],
        cwd=repo,
    )
    print(script, "check_exit", checked.returncode)
    if checked.returncode != 0:
        raise SystemExit(checked.returncode)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-minigrpo")
completed = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_minigrpo.py"),
        "--backend",
        "torch",
        "--model",
        os.environ["MODEL"],
        "--revision",
        os.environ["REVISION"],
        "--steps",
        "40",
        "--group-size",
        "4",
        "--max-new-tokens",
        "32",
        "--epochs",
        "1",
        "--learning-rate",
        "1e-5",
        "--num-layers",
        "1",
        "--kl-coef",
        "0",
        "--seed",
        "20261022",
        "--output",
        str(minigrpo_output),
    ],
    cwd=repo,
)
print("minigrpo_exit", completed.returncode)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-minigrpo")
completed = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_selfimprove.py"),
        "--backend",
        "torch",
        "--model",
        os.environ["MODEL"],
        "--revision",
        os.environ["REVISION"],
        "--rounds",
        "2",
        "--steps-per-round",
        "8",
        "--problems",
        "8",
        "--group-size",
        "2",
        "--max-new-tokens",
        "32",
        "--learning-rate",
        "1e-5",
        "--num-layers",
        "1",
        "--reward",
        "correctness",
        "--seed",
        "20261023",
        "--output",
        str(selfimprove_output),
    ],
    cwd=repo,
)
print("selfimprove_exit", completed.returncode)
